# BulkFormer-Latest: contextual gene features

Latest encodes the complete **19,973-gene axis** and returns a **768-dimensional
vector per gene**, together with expression predictions. This notebook extracts
selected gene vectors after full-transcriptome encoding, pools sample features,
and demonstrates recovery of masked expression values.

Run the installation commands in the repository README first. Select your GPU
before starting Jupyter, for example with `CUDA_VISIBLE_DEVICES=0 jupyter lab`.
The displayed GPU outputs use eight **synthetic** count profiles. They demonstrate
the workflow and are not a measure of biological prediction accuracy.


In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch

if Path.cwd().name == "notebooks":
    os.chdir("..")

device = "cuda"
assert torch.cuda.is_available(), "This tutorial uses a CUDA GPU."
torch.set_num_threads(4)
print(torch.cuda.get_device_name(0))


NVIDIA A800 80GB PCIe


### Load the model and gene resources

Download the Google Drive folder linked in the README into `model/latest/`
before running the notebook, or uncomment the download line below. It contains
one complete `model.pt`, the model configuration, gene vocabulary and graph.
The loader checks file integrity before loading the model.


In [2]:
from bulkformer import download_model, load_latest
from bulkformer import prepare_latest_input, extract_latest

model_dir = Path("model/latest")
# download_model("latest", model_dir)  # Run once if resources are not downloaded.
model, gene_info = load_latest(model_dir, device=device)
print(model.release_metadata["checkpoint_id"])
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
print(f"{len(gene_info):,} coding genes")


src/bulkformer/latest.py:54: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  ).coalesce().to_sparse_csr()


latest-20261104-epoch19-update140885
154,551,299 parameters
19,973 coding genes


### Prepare full-transcriptome input

Rows are samples; columns are human Ensembl gene IDs. Start with the **complete
available coding transcriptome**, including genes with measured zero expression.
Unavailable entries may be omitted or represented with `NaN`; preprocessing maps
them to the model's `-10` sentinel. Measured zeros remain zero.

Counts are normalized over the matched coding genes and transformed to **natural
ln(TPM + 1)**. For already normalized data, choose `input_scale="tpm"`,
`"log1p_tpm"`, or `"log2_tpm"` explicitly. Atlas and Latest use different log bases.


In [3]:
count_df = pd.read_csv("examples/demo_counts.csv", index_col=0)
print("Count matrix:", count_df.shape)
count_df.iloc[:5, :6]


Count matrix: (8, 41671)


,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938
demo_01,123,330,66,111,565,185
demo_02,185,131,365,68,304,197
demo_03,153,101,357,147,246,161
demo_04,112,86,152,91,92,58
demo_05,151,339,163,96,376,234


In [4]:
expression, coverage = prepare_latest_input(
    count_df, gene_info, input_scale="counts"
)
print("Model input:", expression.shape)
coverage


Model input: (8, 19973)


,observed_genes,missing_fraction
demo_01,19973,0.0
demo_02,19973,0.0
demo_03,19973,0.0
demo_04,19973,0.0
demo_05,19973,0.0
demo_06,19973,0.0
demo_07,19973,0.0
demo_08,19973,0.0


### Extract contextual gene embeddings

We request five gene outputs while still passing **all 19,973 input positions**.
For HVGs or DEGs, follow the same order: encode the full input, select the desired
contextual gene vectors, then pool if sample features are needed. Do not truncate
the input matrix to those genes.

The sample vectors returned here mean-pool all observed coding genes, including
measured zeros. Atlas is the recommended starting point for sample-level prediction.


In [5]:
interested_genes = ["TP53", "EGFR", "BRCA1", "GAPDH", "ACTB"]
gene_indices = gene_info.index[gene_info["symbol"].isin(interested_genes)].to_numpy()
selected_genes = gene_info.iloc[gene_indices][["ensembl_gene", "symbol"]]
selected_genes


,ensembl_gene,symbol
308,ENSG00000012048,BRCA1
1308,ENSG00000075624,ACTB
3876,ENSG00000111640,GAPDH
8009,ENSG00000141510,TP53
8739,ENSG00000146648,EGFR


In [6]:
features = extract_latest(
    model, expression, gene_indices=gene_indices,
    return_expression=True, batch_size=1, progress=False
)
gene_embeddings = features["gene_embeddings"]
sample_embeddings = features["sample_embeddings"]
predicted_expression = features["expression_prediction"]
print("Gene embeddings:", gene_embeddings.shape)
print("Sample embeddings:", sample_embeddings.shape)
print("Expression predictions:", predicted_expression.shape)
print("All values finite:", all(np.isfinite(x).all() for x in features.values()))
pd.DataFrame(gene_embeddings[0, :, :6], index=selected_genes["symbol"],
             columns=[f"feature_{i}" for i in range(6)])


Gene embeddings: (8, 5, 768)
Sample embeddings: (8, 768)
Expression predictions: (8, 19973)
All values finite: True


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5
symbol,,,,,,
BRCA1,0.015329,-0.039887,0.522938,4.106353,0.604067,-0.296950
ACTB,-0.162923,0.282975,0.680863,0.100534,-0.286176,-0.341571
GAPDH,-0.054150,-0.828455,0.270986,0.370157,-0.532871,-0.688957
TP53,-0.148930,0.108922,0.210668,2.907683,0.471192,-0.376262
EGFR,0.090245,-0.515544,0.207334,4.537467,-0.020336,-0.179073


### Pool a selected gene set

This example pools the five requested outputs. Replace the gene list with your
HVGs or DEGs for an application-specific representation. Only observed members
of the set contribute to a sample's mean. Using a fixed gene set across samples
makes the representation easier to compare.


In [7]:
observed = expression[:, gene_indices] >= 0
assert observed.any(axis=1).all(), "Each sample needs an observed gene in the pooling set."
subset_embeddings = (gene_embeddings * observed[:, :, None]).sum(axis=1)
subset_embeddings /= observed.sum(axis=1, keepdims=True)
print("Selected-gene pooled embeddings:", subset_embeddings.shape)
pd.DataFrame(subset_embeddings[:, :6], index=count_df.index,
             columns=[f"feature_{i}" for i in range(6)])


Selected-gene pooled embeddings: (8, 768)


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5
demo_01,-0.052086,-0.198398,0.378558,2.404439,0.047175,-0.376563
demo_02,0.020824,-0.267755,0.475978,2.041018,0.170681,-0.368685
demo_03,0.054973,-0.073383,0.462446,3.089135,0.186360,-0.331634
demo_04,0.083653,-0.204475,0.393171,3.055305,0.147873,-0.320176
demo_05,0.086778,-0.192453,0.449675,3.112795,0.179995,-0.285969
demo_06,0.018643,-0.165926,0.412315,2.684926,0.046554,-0.341744
demo_07,0.074850,-0.272262,0.434218,2.316738,0.106309,-0.323159
demo_08,0.023761,-0.260380,0.410926,1.902335,0.068268,-0.264862


### Recover masked expression

For this demonstration, hide 15% of a profile, including the five genes above.
The model measures the actual missing fraction from the sentinel values and
uses it to calibrate the prediction head. No separate `mask_ratio` argument is needed.
Recovery at substantial missingness should be evaluated on the intended data.

Predictions are on the **natural ln(TPM + 1)** scale and may be slightly negative.
GPU sparse graph operations can introduce small numerical differences between runs.


In [8]:
rng = np.random.default_rng(42)
available = np.flatnonzero(expression[0] >= 0)
remaining = np.setdiff1d(available, gene_indices)
n_hidden = round(0.15 * len(available))
hidden_genes = np.r_[gene_indices,
                     rng.choice(remaining, n_hidden - len(gene_indices), replace=False)]
masked_expression = expression[:1].copy()
masked_expression[0, hidden_genes] = -10
print(f"Missing fraction: {(masked_expression == -10).mean():.2%}")

recovered = extract_latest(model, masked_expression, return_expression=True, progress=False)
pd.DataFrame({
    "symbol": selected_genes["symbol"].to_numpy(),
    "held_out_ln_tpm": expression[0, gene_indices],
    "predicted_ln_tpm": recovered["expression_prediction"][0, gene_indices],
})


Missing fraction: 15.00%


,symbol,held_out_ln_tpm,predicted_ln_tpm
0,BRCA1,1.045305,2.701303
1,ACTB,3.391354,3.841492
2,GAPDH,4.692566,3.855397
3,TP53,1.825565,2.875112
4,EGFR,2.748207,2.930731


### Save the outputs with their identifiers


In [9]:
output_dir = Path("outputs/latest_demo")
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_dir / "features.npz",
    sample_ids=count_df.index.to_numpy(dtype=str),
    selected_gene_ids=selected_genes["ensembl_gene"].to_numpy(dtype=str),
    expression_gene_ids=gene_info["ensembl_gene"].to_numpy(dtype=str),
    gene_embeddings=gene_embeddings,
    sample_embeddings=sample_embeddings,
    selected_gene_pooled_embeddings=subset_embeddings,
    expression_prediction=predicted_expression,
    checkpoint_id=model.release_metadata["checkpoint_id"],
)
print(output_dir / "features.npz")


outputs/latest_demo/features.npz


See the [model card](../docs/models/latest.md) for architecture and checkpoint details,
and the [input guide](../docs/input-guide.md) for coverage and normalization conventions.
